In [1]:
import pandas as pd
import time
from web3 import Web3

# Load all CSV files
shipment_df = pd.read_csv("shipment_registry.csv")
gps_df      = pd.read_csv("gps_readings.csv")
rfid_df     = pd.read_csv("rfid_readings.csv")
temp_df     = pd.read_csv("temperature_readings.csv")
iot_df      = pd.read_csv("iot_data.csv")

print(f"Shipments:    {len(shipment_df)} rows")
print(f"GPS:          {len(gps_df)} rows")
print(f"RFID:         {len(rfid_df)} rows")
print(f"Temperature:  {len(temp_df)} rows")
print(f"IoT combined: {len(iot_df)} rows")
print()
print(iot_df.head())

Shipments:    30 rows
GPS:          150 rows
RFID:         84 rows
Temperature:  95 rows
IoT combined: 329 rows

     reading_id  rfid_tag device_id    data_type            data_value  \
0  RDG-TMP-0031  RFID-011    TMP104  Temperature                  12.2   
1  RDG-GPS-0096  RFID-020    GPS291          GPS  14.601956,120.989036   
2  RDG-GPS-0051  RFID-011    GPS728          GPS  14.620032,120.962165   
3  RDG-RFD-0027  RFID-011    RFD702         RFID              VERIFIED   
4  RDG-GPS-0097  RFID-020    GPS920          GPS  14.595757,120.981906   

             timestamp    gps_lat     gps_lng  temperature_c  
0  2026-05-03 07:00:00        NaN         NaN           12.2  
1  2026-05-03 07:00:00  14.601956  120.989036            NaN  
2  2026-05-03 07:00:00  14.620032  120.962165            NaN  
3  2026-05-03 08:00:00        NaN         NaN            NaN  
4  2026-05-03 09:00:00  14.595757  120.981906            NaN  


In [2]:
ganache_url = "http://127.0.0.1:8545"
w3 = Web3(Web3.HTTPProvider(ganache_url))

if w3.is_connected():
    print("Connection successful!")
else:
    print("Connection failed. Check if Ganache is running.")

Connection successful!


In [6]:
import json

contract_address = Web3.to_checksum_address("0x046A632CF5E94b457D33b7B8c7052b4ACA05693C")

abi_string = '''[
	{
		"inputs": [],
		"stateMutability": "nonpayable",
		"type": "constructor"
	},
	{
		"anonymous": false,
		"inputs": [
			{
				"indexed": false,
				"internalType": "uint256",
				"name": "timestamp",
				"type": "uint256"
			},
			{
				"indexed": true,
				"internalType": "string",
				"name": "rfidTag",
				"type": "string"
			},
			{
				"indexed": false,
				"internalType": "string",
				"name": "deviceId",
				"type": "string"
			},
			{
				"indexed": false,
				"internalType": "string",
				"name": "dataType",
				"type": "string"
			},
			{
				"indexed": false,
				"internalType": "string",
				"name": "dataValue",
				"type": "string"
			}
		],
		"name": "DataStored",
		"type": "event"
	},
	{
		"anonymous": false,
		"inputs": [
			{
				"indexed": false,
				"internalType": "uint256",
				"name": "timestamp",
				"type": "uint256"
			},
			{
				"indexed": true,
				"internalType": "string",
				"name": "rfidTag",
				"type": "string"
			},
			{
				"indexed": false,
				"internalType": "string",
				"name": "deviceId",
				"type": "string"
			},
			{
				"indexed": false,
				"internalType": "string",
				"name": "latitude",
				"type": "string"
			},
			{
				"indexed": false,
				"internalType": "string",
				"name": "longitude",
				"type": "string"
			}
		],
		"name": "LocationStored",
		"type": "event"
	},
	{
		"inputs": [
			{
				"internalType": "string",
				"name": "rfidTag",
				"type": "string"
			},
			{
				"internalType": "enum IoTDataStorage.GoodsCategory",
				"name": "category",
				"type": "uint8"
			},
			{
				"internalType": "string",
				"name": "origin",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "destination",
				"type": "string"
			},
			{
				"internalType": "uint16",
				"name": "packageCount",
				"type": "uint16"
			},
			{
				"internalType": "string",
				"name": "vehicleId",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "driverId",
				"type": "string"
			}
		],
		"name": "registerShipment",
		"outputs": [],
		"stateMutability": "nonpayable",
		"type": "function"
	},
	{
		"anonymous": false,
		"inputs": [
			{
				"indexed": false,
				"internalType": "uint256",
				"name": "timestamp",
				"type": "uint256"
			},
			{
				"indexed": true,
				"internalType": "string",
				"name": "rfidTag",
				"type": "string"
			},
			{
				"indexed": false,
				"internalType": "string",
				"name": "deviceId",
				"type": "string"
			},
			{
				"indexed": false,
				"internalType": "string",
				"name": "status",
				"type": "string"
			}
		],
		"name": "RFIDScanned",
		"type": "event"
	},
	{
		"anonymous": false,
		"inputs": [
			{
				"indexed": true,
				"internalType": "string",
				"name": "rfidTag",
				"type": "string"
			},
			{
				"indexed": false,
				"internalType": "string",
				"name": "origin",
				"type": "string"
			},
			{
				"indexed": false,
				"internalType": "string",
				"name": "destination",
				"type": "string"
			},
			{
				"indexed": false,
				"internalType": "uint8",
				"name": "category",
				"type": "uint8"
			}
		],
		"name": "ShipmentRegistered",
		"type": "event"
	},
	{
		"inputs": [
			{
				"internalType": "string",
				"name": "readingId",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "rfidTag",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "deviceId",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "deviceType",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "dataType",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "dataValue",
				"type": "string"
			}
		],
		"name": "storeData",
		"outputs": [],
		"stateMutability": "nonpayable",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "string",
				"name": "rfidTag",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "deviceId",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "latitude",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "longitude",
				"type": "string"
			}
		],
		"name": "storeGPS",
		"outputs": [],
		"stateMutability": "nonpayable",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "string",
				"name": "rfidTag",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "deviceId",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "status",
				"type": "string"
			}
		],
		"name": "storeRFIDScan",
		"outputs": [],
		"stateMutability": "nonpayable",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "string",
				"name": "rfidTag",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "deviceId",
				"type": "string"
			},
			{
				"internalType": "int16",
				"name": "tempTimes10",
				"type": "int16"
			}
		],
		"name": "storeTemperature",
		"outputs": [],
		"stateMutability": "nonpayable",
		"type": "function"
	},
	{
		"anonymous": false,
		"inputs": [
			{
				"indexed": false,
				"internalType": "uint256",
				"name": "timestamp",
				"type": "uint256"
			},
			{
				"indexed": true,
				"internalType": "string",
				"name": "rfidTag",
				"type": "string"
			},
			{
				"indexed": false,
				"internalType": "string",
				"name": "deviceId",
				"type": "string"
			},
			{
				"indexed": false,
				"internalType": "int16",
				"name": "tempTimes10",
				"type": "int16"
			}
		],
		"name": "TemperatureStored",
		"type": "event"
	},
	{
		"inputs": [
			{
				"internalType": "address",
				"name": "newOwner",
				"type": "address"
			}
		],
		"name": "transferOwnership",
		"outputs": [],
		"stateMutability": "nonpayable",
		"type": "function"
	},
	{
		"inputs": [],
		"name": "getAllRFIDTags",
		"outputs": [
			{
				"internalType": "string[]",
				"name": "",
				"type": "string[]"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "string",
				"name": "rfidTag",
				"type": "string"
			}
		],
		"name": "getLocationsByRFID",
		"outputs": [
			{
				"components": [
					{
						"internalType": "uint256",
						"name": "timestamp",
						"type": "uint256"
					},
					{
						"internalType": "string",
						"name": "rfidTag",
						"type": "string"
					},
					{
						"internalType": "string",
						"name": "deviceId",
						"type": "string"
					},
					{
						"internalType": "string",
						"name": "latitude",
						"type": "string"
					},
					{
						"internalType": "string",
						"name": "longitude",
						"type": "string"
					},
					{
						"internalType": "address",
						"name": "recordedBy",
						"type": "address"
					}
				],
				"internalType": "struct IoTDataStorage.GPSRecord[]",
				"name": "",
				"type": "tuple[]"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "string",
				"name": "rfidTag",
				"type": "string"
			}
		],
		"name": "getRFIDScansByRFID",
		"outputs": [
			{
				"components": [
					{
						"internalType": "uint256",
						"name": "timestamp",
						"type": "uint256"
					},
					{
						"internalType": "string",
						"name": "rfidTag",
						"type": "string"
					},
					{
						"internalType": "string",
						"name": "deviceId",
						"type": "string"
					},
					{
						"internalType": "string",
						"name": "status",
						"type": "string"
					},
					{
						"internalType": "address",
						"name": "recordedBy",
						"type": "address"
					}
				],
				"internalType": "struct IoTDataStorage.RFIDRecord[]",
				"name": "",
				"type": "tuple[]"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "string",
				"name": "rfidTag",
				"type": "string"
			}
		],
		"name": "getShipment",
		"outputs": [
			{
				"components": [
					{
						"internalType": "string",
						"name": "rfidTag",
						"type": "string"
					},
					{
						"internalType": "enum IoTDataStorage.GoodsCategory",
						"name": "category",
						"type": "uint8"
					},
					{
						"internalType": "string",
						"name": "origin",
						"type": "string"
					},
					{
						"internalType": "string",
						"name": "destination",
						"type": "string"
					},
					{
						"internalType": "uint16",
						"name": "packageCount",
						"type": "uint16"
					},
					{
						"internalType": "string",
						"name": "vehicleId",
						"type": "string"
					},
					{
						"internalType": "string",
						"name": "driverId",
						"type": "string"
					},
					{
						"internalType": "uint256",
						"name": "registeredAt",
						"type": "uint256"
					},
					{
						"internalType": "bool",
						"name": "exists",
						"type": "bool"
					}
				],
				"internalType": "struct IoTDataStorage.Shipment",
				"name": "",
				"type": "tuple"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "string",
				"name": "rfidTag",
				"type": "string"
			}
		],
		"name": "getTemperaturesByRFID",
		"outputs": [
			{
				"components": [
					{
						"internalType": "uint256",
						"name": "timestamp",
						"type": "uint256"
					},
					{
						"internalType": "string",
						"name": "rfidTag",
						"type": "string"
					},
					{
						"internalType": "string",
						"name": "deviceId",
						"type": "string"
					},
					{
						"internalType": "int16",
						"name": "tempTimes10",
						"type": "int16"
					},
					{
						"internalType": "address",
						"name": "recordedBy",
						"type": "address"
					}
				],
				"internalType": "struct IoTDataStorage.TempRecord[]",
				"name": "",
				"type": "tuple[]"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [],
		"name": "gpsRecordCount",
		"outputs": [
			{
				"internalType": "uint256",
				"name": "",
				"type": "uint256"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "",
				"type": "uint256"
			}
		],
		"name": "gpsRecords",
		"outputs": [
			{
				"internalType": "uint256",
				"name": "timestamp",
				"type": "uint256"
			},
			{
				"internalType": "string",
				"name": "rfidTag",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "deviceId",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "latitude",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "longitude",
				"type": "string"
			},
			{
				"internalType": "address",
				"name": "recordedBy",
				"type": "address"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [],
		"name": "iotRecordCount",
		"outputs": [
			{
				"internalType": "uint256",
				"name": "",
				"type": "uint256"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "",
				"type": "uint256"
			}
		],
		"name": "iotRecords",
		"outputs": [
			{
				"internalType": "uint256",
				"name": "timestamp",
				"type": "uint256"
			},
			{
				"internalType": "string",
				"name": "readingId",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "rfidTag",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "deviceId",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "deviceType",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "dataType",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "dataValue",
				"type": "string"
			},
			{
				"internalType": "address",
				"name": "recordedBy",
				"type": "address"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [],
		"name": "MAX_ENTRIES",
		"outputs": [
			{
				"internalType": "uint256",
				"name": "",
				"type": "uint256"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [],
		"name": "owner",
		"outputs": [
			{
				"internalType": "address",
				"name": "",
				"type": "address"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [],
		"name": "rfidRecordCount",
		"outputs": [
			{
				"internalType": "uint256",
				"name": "",
				"type": "uint256"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "",
				"type": "uint256"
			}
		],
		"name": "rfidRecords",
		"outputs": [
			{
				"internalType": "uint256",
				"name": "timestamp",
				"type": "uint256"
			},
			{
				"internalType": "string",
				"name": "rfidTag",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "deviceId",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "status",
				"type": "string"
			},
			{
				"internalType": "address",
				"name": "recordedBy",
				"type": "address"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "",
				"type": "uint256"
			}
		],
		"name": "rfidTags",
		"outputs": [
			{
				"internalType": "string",
				"name": "",
				"type": "string"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [],
		"name": "shipmentCount",
		"outputs": [
			{
				"internalType": "uint256",
				"name": "",
				"type": "uint256"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [],
		"name": "tempRecordCount",
		"outputs": [
			{
				"internalType": "uint256",
				"name": "",
				"type": "uint256"
			}
		],
		"stateMutability": "view",
		"type": "function"
	},
	{
		"inputs": [
			{
				"internalType": "uint256",
				"name": "",
				"type": "uint256"
			}
		],
		"name": "tempRecords",
		"outputs": [
			{
				"internalType": "uint256",
				"name": "timestamp",
				"type": "uint256"
			},
			{
				"internalType": "string",
				"name": "rfidTag",
				"type": "string"
			},
			{
				"internalType": "string",
				"name": "deviceId",
				"type": "string"
			},
			{
				"internalType": "int16",
				"name": "tempTimes10",
				"type": "int16"
			},
			{
				"internalType": "address",
				"name": "recordedBy",
				"type": "address"
			}
		],
		"stateMutability": "view",
		"type": "function"
	}
]'''

contract_abi = json.loads(abi_string)
contract = w3.eth.contract(address=contract_address, abi=contract_abi)
w3.eth.default_account = w3.eth.accounts[0]

print(f"Connected to Smart Contract at {contract_address}")
print(f"Owner: {contract.functions.owner().call()}")

Connected to Smart Contract at 0x046A632CF5E94b457D33b7B8c7052b4ACA05693C
Owner: 0x929005F082e4E8B3275D7eb7D60bb603dd99b05f


In [7]:
# Register all shipments
# GoodsCategory index mapping
CATEGORY_MAP = {
    "Deep Freeze": 0, "Frozen": 1, "Chill/Refrigerated": 2,
    "Pharma": 3, "Cool-Chain": 4, "Dry Goods": 5,
    "Electronics": 6, "Clothing": 7, "Industrial": 8
}

print("Registering shipments...")
for _, row in shipment_df.iterrows():
    try:
        txn = contract.functions.registerShipment(
            str(row["rfid_tag"]),
            CATEGORY_MAP.get(row["goods_category"], 5),
            str(row["origin"]),
            str(row["destination"]),
            int(row["package_count"]),
            str(row["vehicle_id"]),
            str(row["driver_id"])
        ).transact({'from': w3.eth.default_account, 'gas': 1000000})
        w3.eth.wait_for_transaction_receipt(txn)
        print(f"  Registered {row['rfid_tag']}")
    except Exception as e:
        print(f"  Skipped {row['rfid_tag']}: {e}")

print(f"\nDone. Total shipments registered: {contract.functions.shipmentCount().call()}")

Registering shipments...
  Registered RFID-001
  Registered RFID-002
  Registered RFID-003
  Registered RFID-004
  Registered RFID-005
  Registered RFID-006
  Registered RFID-007
  Registered RFID-008
  Registered RFID-009
  Registered RFID-010
  Registered RFID-011
  Registered RFID-012
  Registered RFID-013
  Registered RFID-014
  Registered RFID-015
  Registered RFID-016
  Registered RFID-017
  Registered RFID-018
  Registered RFID-019
  Registered RFID-020
  Registered RFID-021
  Registered RFID-022
  Registered RFID-023
  Registered RFID-024
  Registered RFID-025
  Registered RFID-026
  Registered RFID-027
  Registered RFID-028
  Registered RFID-029
  Registered RFID-030

Done. Total shipments registered: 31


In [8]:
# Send IoT data to blockchain
def send_iot_data(reading_id, rfid_tag, device_id, device_type, data_type, data_value):
    """Sends one IoT reading to the smart contract"""
    txn = contract.functions.storeData(
        str(reading_id),
        str(rfid_tag),
        str(device_id),
        str(device_type),
        str(data_type),
        str(data_value)
    ).transact({'from': w3.eth.default_account, 'gas': 1000000})
    receipt = w3.eth.wait_for_transaction_receipt(txn)
    print(f"  Stored {data_type} | {rfid_tag} | {data_value} | Txn: {receipt.transactionHash.hex()}")

print("Sending IoT data to blockchain...")
for _, row in iot_df.iterrows():
    try:
        send_iot_data(
            row["reading_id"],
            row["rfid_tag"],
            row["device_id"],
            row["data_type"], # deviceType
            row["data_type"], # dataType
            row["data_value"]
        )
        time.sleep(0.5)
    except Exception as e:
        print(f"  Skipped {row['reading_id']}: {e}")

Sending IoT data to blockchain...
  Stored Temperature | RFID-011 | 12.2 | Txn: 7b294f28c4e394dbc6379d0241c002488e7df1cfd419fdaa2364812f44b48126
  Stored GPS | RFID-020 | 14.601956,120.989036 | Txn: ca19de2a53abfe6a87aef1967ee39008a30cfc829fa852d1738fb377b862a3ae
  Stored GPS | RFID-011 | 14.620032,120.962165 | Txn: 52ac54df8e30cdc2281c4782f0bd5272dffa3f1e5d3a9977bbb7693416f6af5a
  Stored RFID | RFID-011 | VERIFIED | Txn: d29b1da998efc7eb9b38f682df136fdd99e866637fa99b2c92bbe723202b826e
  Stored GPS | RFID-020 | 14.595757,120.981906 | Txn: 3082c01533d803b4b337270f2481cbb03bf0d9d53f8431d9ccd08196ca236afa
  Stored GPS | RFID-028 | 14.374937,121.038127 | Txn: f7768aeb79e3c47ffd0421b64b59e13dfa8224043868af3b1e794b3a240e0e6c
  Stored RFID | RFID-020 | VERIFIED | Txn: cebd49e413fe7a8d26355bc9e055621aa895b5a3d587481ed5e5413a2db0a8e2
  Stored Temperature | RFID-011 | 13.6 | Txn: 0bf55561c06991bf933090eeaedd72dff3c611912b044e40d3bc919f06ff0041
  Stored GPS | RFID-011 | 14.622406,120.970944 | Txn

In [10]:
# Verify retrieved data
print("Blockchain Verification")
print(f"Shipments registered: {contract.functions.shipmentCount().call()}")
print(f"IoT records stored:   {contract.functions.iotRecordCount().call()}")
print(f"GPS records:          {contract.functions.gpsRecordCount().call()}")
print(f"Temp records:         {contract.functions.tempRecordCount().call()}")
print(f"RFID records:         {contract.functions.rfidRecordCount().call()}")

# Retrieve first shipment
first_tag = shipment_df.iloc[0]["rfid_tag"]
shipment = contract.functions.getShipment(first_tag).call()
print(f"\nFirst Shipment ({first_tag}):")
print(f"Origin:      {shipment[2]}")
print(f"Destination: {shipment[3]}")
print(f"Category:    {shipment[1]}")
print(f"Packages:    {shipment[4]}")

Blockchain Verification
Shipments registered: 31
IoT records stored:   330
GPS records:          0
Temp records:         0
RFID records:         0

First Shipment (RFID-001):
Origin:      General Mariano Alvarez
Destination: San Andres
Category:    3
Packages:    16
